# Dunnhumby · GraphSAGE · 남은 9시드 → 10시드 최종 집계

**학교 계정 jungunb@u.sogang.ac.kr로 Colab과 Drive에 로그인하세요. 개인 계정의 실행과 별도 저장됩니다.**

**기존 GraphSAGE seed49의 8개 결과를 재사용하고, seed42~48·50·51의 8개 모델만 순차 학습합니다. 총 새 학습은 72개입니다.**

| 순서 | 모델 | 구성 |
|---|---|---|
|1|M1|GraphSAGE 기준모형|
|2|M2|임베딩·표현 개입|
|3|M3|N/V 가치그래프 개입|
|4|M4|고객 내 손실 가중치|
|5|M5_A|M2+M3|
|6|M5_B|M2+M4|
|7|M5_C|M3+M4|
|8|M5_D|M2+M3+M4|

- 최종 패널: **8모델 × seed42~51 = 80개 결과**. seed49는 재학습·재평가하지 않습니다. 다른 seed는 seed49 체크포인트로 초기화하지 않고 각 seed에서 새로 시작합니다.
- 원래 GraphSAGE 조건 유지: L4, ID64, 2층(각64), 0/1/2층 평균64차원, ELU 및 층별 L2 정규화, dropout 없음, 전체 이웃 평균. batch8192, Adam lr.0005, ID L2 .001, M2 공유표현 및 백본 L2 1e-6, uniform 음성표본1개.
- train DAY1~704 / test DAY705~711, 신규상품 추천(학습 구매쌍 정답·후보 제외), MIN_ITEM_INTER=1. CLV 입력은 기존 학습구간 내365일 historical proxy.
- 확정 M2~M4 계산과 한 optimizer 공동학습을 유지합니다. **각300epoch 마지막점만 test 평가**. validation/holdout/중간 test/조기종료/순열 추가 학습 없음.
- seed49 원본 SHA와 입력·설정을 먼저 검증합니다. 다른 파일이 섞이거나 원본이 없으면 자동 재학습하지 않고 중단합니다.
- 실행을 시작한 뒤에는 **이 고정 링크, 참조 경로, 출력 경로를 그대로 유지**하세요. 같은 출력 폴더를 두 GPU에서 동시에 실행하지 마세요. 다른 GPU의 실험은 건드릴 필요가 없습니다.

## 저장 및 재개
모델·Adam·numpy/torch/CUDA 난수 상태를 **매 epoch Drive에 저장 완료한 뒤** 다음 epoch로 진행합니다. 중단 시 같은 노트북을 위에서부터 재실행하면 완료 모델은 건너뛰고 미완료 모델은 마지막 저장 epoch 다음부터 이어갑니다. 중간에 끊긴 epoch만 다시 계산할 수 있습니다. 손상된 저장본이나 Drive 오류는 보존하고 멈춥니다.

기존 최적화(고정 입력 장치 캐시, 로그용 GPU 동기화 축소, 로컬 직렬화 후 Drive 동기 복사, 초기 시간 계측)를 유지합니다. 학습 임베딩은 매 배치 다시 계산합니다. AMP/배치 확대/샘플 변경은 하지 않습니다. 평가 후 결과 파일 저장 직전에 끊긴 경우에는 평가만 다시 실행될 수 있습니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys, torch
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
assert torch.cuda.is_available(), '런타임 유형을 L4 GPU로 설정하세요.'
GPU = torch.cuda.get_device_name(0)
assert 'L4' in GPU, f'현재 GPU: {GPU}. 기존 조건과 같은 L4로 실행하세요.'
SOURCE_COMMIT = 'cb5a805306b71302d4af79184745803561486c09'
REPO = Path('/content/clv-dh-graphsage-remaining9-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', SOURCE_COMMIT], check=True)
actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert actual == SOURCE_COMMIT
sys.path.insert(0, str(REPO))
print('GPU:', GPU, '| 고정 코드:', SOURCE_COMMIT)

## 1. 학교 Drive 입력 준비

학교 MyDrive 최상위에 준비해 둔 `graphsage_school_inputs.zip`을 검증하고 풉니다. 데이터·참조 파일의 SHA256을 검사하며 같은 이름의 다른 파일을 덮어쓰지 않습니다. 입력 준비 셀에서는 학습하지 않습니다.


In [ ]:
from zipfile import ZipFile
import hashlib, shutil
BUNDLE = Path('/content/drive/MyDrive/graphsage_school_inputs.zip')
assert BUNDLE.is_file(), '학교 Drive가 연결됐는지 확인하세요. MyDrive에 graphsage_school_inputs.zip이 필요합니다.'
DATA = Path('/content/drive/MyDrive/논문/data')
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()
with ZipFile(BUNDLE) as z:
    manifest = json.loads(z.read('manifest.json'))
    assert len(manifest) == 5
    for name, expected in manifest.items():
        target = (DATA / name).resolve()
        assert DATA.resolve() in target.parents, '잘못된 ZIP 경로'
        if target.exists():
            assert sha256_file(target) == expected, f'기존 파일 불일치: {target}'
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        temp = target.with_suffix(target.suffix + '.partial')
        with z.open(name) as src, open(temp, 'wb') as dst:
            shutil.copyfileobj(src, dst)
        assert sha256_file(temp) == expected, f'복사 검증 실패: {name}'
        temp.replace(target)
print('학교 Drive 입력 5개 준비·검증 완료')

import clv_dh_graphsage_remaining9 as screen
assert Path(screen.__file__).resolve().parent == REPO.resolve(), '구버전 모듈이 메모리에 남았습니다. 이 런타임만 다시 시작하세요.'
assert Path(screen.original.__file__).resolve().parent == REPO.resolve(), '기존 GraphSAGE 모듈이 다른 경로입니다. 이 런타임만 다시 시작하세요.'
cfg = screen.configure(
    seed49_reference=str(DATA / 'references/dh_graphsage_final8_seed49_results_recovered.zip'),
    m2_reference=str(DATA / 'references/clv_m2_adaptive_dunnhumby_final10_results.zip'),
    m4_reference=str(DATA / 'references/clv_m4_user_centered_final10_results.zip'),
    out_dir=str(DATA / 'results_v3_dunnhumby_graphsage_final8_final10_school_v1'),
)
print('seed49 원본:', cfg.seed49_reference)
print('M2 입력 참조:', cfg.m2_reference)
print('M3/M4 입력 참조:', cfg.m4_reference)
print('새 결과:', cfg.out_dir)
prepared = screen.prepare(cfg)
assert cfg.seeds == tuple(range(42, 52)) and cfg.epochs == 300
print('새 학습 시드:', screen.NEW_SEEDS)
print('각 시드 순서:', screen.ORDER)
print('진행·checkpoint·결과 저장:', prepared['run_dir'])
print(json.dumps(screen.report(prepared)['reading'], ensure_ascii=False, indent=2))

## 2. 남은 9시드 × 8모델 순차 학습

seed42 → 43 → … → 48 → 50 → 51 순서이며 각 seed에서 M1, M2, M3, M4, M5_A, M5_B, M5_C, M5_D를 실행합니다.
완료 결과는 재학습·재평가 없이 건너뜁니다. 실행 도중 값을 보고 모형·강도·epoch·시드를 재선택하지 않습니다.

매 epoch 저장을 유지하며 로그는 초기 및 일정 간격으로 표시합니다. 로그가 매 epoch 나오지 않아도 저장은 매 epoch 수행합니다. 각 모델 평가 후 전체 완료 현황과 개별 결과를 저장합니다.

In [ ]:
result = screen.run(prepared)
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))
assert result['complete']
assert result['reading']['completed_total'] == 80

## 3. 학습·저장 시간 확인 (선택)

학습이 끝났거나 셀을 중지한 뒤 확인하세요. 첫3개 epoch의 첫5개 배치에 대해 입력/음성표본·학습계산을 별도로 계측하고, 매 epoch의 전체 학습·저장 시간을 기록합니다. 계측은 현재 재개 세션 초반에도 수행됩니다. 이미 실행 중인 학습을 시간표 확인만을 위해 중단할 필요는 없습니다.

In [ ]:
import pandas as pd
timing_paths = sorted((prepared['run_dir'] / 'progress' / 'timing').glob('*.csv'))
if timing_paths:
    timing = pd.concat([pd.read_csv(p).assign(timing_file=p.name) for p in timing_paths], ignore_index=True)
    display(timing.tail(12))
else:
    print('아직 저장된 시간 측정 파일이 없습니다.')

## 4. 결과 확인 — 학습/평가를 다시 하지 않습니다

80개가 모두 완료되면 **모델별 10시드 산술평균과 표본 표준편차(ddof=1)**를 생성합니다. 주요12지표는 한 표로 보여주며 전체142지표(전체/CLV 세그먼트/노출 등)를 CSV로 보존합니다.
- absolute.csv: 80개 모델·시드별 절대 성능
- summary.csv: 모델·지표별 평균, 표준편차, 최소·최대
- core_mean_std.csv: 주요12지표 × 8모델 평균 ± 표준편차
- comparison_vs_m1.csv: 같은 GraphSAGE M1 대비 평균의 상대변화 및 대응 차이 요약
- paired_vs_m1.csv: 같은 seed끼리 짝지은 원값·차이
- diagnostics.csv / result.json / protocol.json: 작동 진단·완료 현황·고정 설정

미완료 상태에서는 개별 완료 결과와 현황만 저장하고 **불균형 시드 수로 최종 10시드 표를 만들지 않습니다.** seed49는 원본 arm JSON의 전체 정밀도 값을 사용합니다(CSV 반올림값 대신). 표준편차는 시드 변동성이며, 95% 신뢰구간이나 유의성 검정 결과가 아닙니다. 검정은 모든 결과 회수 후 별도로 수행합니다.

In [ ]:
result = screen.report(prepared)
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))
if result['complete']:
    display(result['core_mean_std'])
    display(result['comparison_vs_m1'].query('metric in @screen.CORE'))
else:
    print('미완료: 아래는 현재 완료된 개별 결과이며 최종 평균이 아닙니다.')
    display(result['absolute'][['model', 'seed', 'source', *screen.CORE]])

## 5. 결과 ZIP 및 Drive 백업

ZIP에는 결과·집계·원본 seed49·새 모델 결과·학습 이력·시간·설정을 포함합니다. 큰 체크포인트는 ZIP에 넣지 않고 기존 Drive 실행 폴더에 유지합니다.
미완료일 때는 파일명에 _partial을 붙여 최종본과 구분합니다. 브라우저 다운로드가 멈추더라도 **MyDrive 최상위에 백업한 ZIP**을 Drive 웹에서 다운로드할 수 있습니다.

In [ ]:
from google.colab import files
import shutil
result = screen.report(prepared)
output = Path(screen.export_zip(
    prepared,
    '/content/dh_graphsage_final8_final10_results.zip',
    allow_partial=not result['complete'],
))
backup = Path('/content/drive/MyDrive') / output.name
shutil.copy2(output, backup)
assert backup.stat().st_size == output.stat().st_size
print('Drive 백업:', backup)
print(f'크기: {backup.stat().st_size / 1024**2:.2f} MB')
print('완료 여부:', result['complete'])
files.download(str(output))